# NGS Pipeline - Google Colab Notebook
This notebook runs the NGS (Next-Generation Sequencing) pipeline in Google Colab with ngrok integration for public URL access.

**Author**: @saurabhgayali

## Pipeline Overview
The pipeline processes paired-end Illumina sequencing reads through 12 stages:
1. **getting_data** - Download SRA data
2. **fastqc_raw** - Quality control on raw reads
3. **trimming** - Adapter and quality trimming
4. **fastqc_trim** - Quality control on trimmed reads
5. **multiqc_raw** - Aggregate QC report (raw)
6. **multiqc_trim** - Aggregate QC report (trimmed)
7. **reference** - Download reference genome
8. **bwa_alignment** - Align reads to reference
9. **samtools_sort** - Sort and index alignments
10. **variant_calling** - Call variants
11. **consensus** - Generate consensus sequence
12. **cds_extraction** - Extract coding sequences

## 1. Install Miniconda and System Dependencies

In [ ]:
# Install Miniconda
!wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
!bash Miniconda3-latest-Linux-x86_64.sh -b -p /usr/local/miniconda
!rm Miniconda3-latest-Linux-x86_64.sh

# Add conda to PATH
import os
os.environ['PATH'] += ':/usr/local/miniconda/bin'
print('Miniconda installed successfully')

## 2. Create and Activate Conda Environment

In [ ]:
# Create conda environment with all NGS tools
!conda create -y -n ngs_env -c bioconda -c conda-forge python=3.11 \
    sra-tools fastqc multiqc trimmomatic bwa samtools bcftools \
    biopython numpy scipy pandas matplotlib seaborn jupyter

print('Conda environment created successfully')

In [ ]:
# Initialize conda shell
import subprocess
import sys

# Source conda initialization
subprocess.run(['bash', '-c', 'source /usr/local/miniconda/etc/profile.d/conda.sh && conda init bash'], check=False)

# Create activation script
with open('/usr/local/miniconda/envs/ngs_env/activate_ngs.sh', 'w') as f:
    f.write('''#!/bin/bash
source /usr/local/miniconda/etc/profile.d/conda.sh
conda activate ngs_env
''')

print('Conda initialization complete')

## 3. Install Python Packages

In [ ]:
# Install pip packages in the conda environment
!bash -c 'source /usr/local/miniconda/etc/profile.d/conda.sh && conda activate ngs_env && pip install fastapi uvicorn pydantic pyliftover ngrok --quiet'
print('Python packages installed successfully')

## 4. Clone NGS Pipeline Repository

In [ ]:
import os
import subprocess

# Clone repository
work_dir = '/content/ngs_pipeline'
os.makedirs(work_dir, exist_ok=True)
os.chdir(work_dir)

# Clone the NGS pipeline repository
!git clone https://github.com/saurabhgayali/NGS-pipeline-docker.git .

# Create necessary directories
!mkdir -p projects data logs

print(f'Repository cloned to {work_dir}')
print('Directories created: projects, data, logs')

## 5. Download and Setup Ngrok

In [ ]:
import subprocess
import os

# Install ngrok from repository
!apt-get update -qq && apt-get install -y -qq ngrok 2>/dev/null || true

# If apt fails, download from ngrok
if not os.path.exists('/usr/local/bin/ngrok'):
    !wget -q -O /tmp/ngrok.zip https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.zip
    !unzip -q /tmp/ngrok.zip -d /usr/local/bin/
    !rm /tmp/ngrok.zip

# Verify ngrok installation
result = subprocess.run(['ngrok', '--version'], capture_output=True, text=True)
print('Ngrok installed:', result.stdout.strip())

## 6. Setup FastAPI Server with Ngrok

In [ ]:
# Start ngrok in the background and setup FastAPI
import subprocess
import time
import requests
from pathlib import Path

# Function to get ngrok public URL
def get_ngrok_url():
    """Get the public URL from ngrok"""
    try:
        response = requests.get('http://127.0.0.1:4040/api/tunnels')
        tunnels = response.json()['tunnels']
        if tunnels:
            return tunnels[0]['public_url']
    except:
        pass
    return None

# Start ngrok
ngrok_process = subprocess.Popen(['ngrok', 'http', '7860', '--log=stdout'], 
                                  stdout=subprocess.PIPE, stderr=subprocess.PIPE)

# Wait for ngrok to start
time.sleep(3)

# Get and display URL
ngrok_url = get_ngrok_url()
if ngrok_url:
    print(f"\n{'='*60}")
    print(f"✓ Ngrok tunnel established!")
    print(f"  Public URL: {ngrok_url}")
    print(f"{'='*60}\n")
else:
    print('Ngrok starting up... (Check back in a moment)')

## 7. Create Colab-Adapted FastAPI Application

In [ ]:
from fastapi import FastAPI, HTTPException
from fastapi.staticfiles import StaticFiles
from fastapi.responses import JSONResponse
import uvicorn
import os
from pathlib import Path
import subprocess
import json
from datetime import datetime

# Create FastAPI app
app = FastAPI(
    title='NGS Pipeline - Colab Edition',
    description='Next-Generation Sequencing Analysis Pipeline',
    version='1.0.0'
)

# Base directory for projects
PROJECTS_DIR = Path('/content/ngs_pipeline/projects')
DATA_DIR = Path('/content/ngs_pipeline/data')

# Pipeline stages
PIPELINE_STAGES = [
    'getting_data',
    'fastqc_raw',
    'trimming',
    'fastqc_trim',
    'multiqc_raw',
    'multiqc_trim',
    'reference',
    'bwa_alignment',
    'samtools_sort',
    'variant_calling',
    'consensus',
    'cds_extraction'
]

@app.get('/api/health')
async def health_check():
    return {'status': 'ok', 'service': 'NGS Pipeline'}

@app.get('/api/stages')
async def get_stages():
    return {'stages': PIPELINE_STAGES}

@app.get('/api/projects')
async def list_projects():
    """List all projects"""
    projects = []
    if PROJECTS_DIR.exists():
        for proj_dir in PROJECTS_DIR.iterdir():
            if proj_dir.is_dir():
                projects.append(proj_dir.name)
    return {'projects': projects}

print('FastAPI app created successfully')

## 8. Start FastAPI Server

In [ ]:
import subprocess
import time
import threading

# Create a script to run the server
server_script = '''
from fastapi import FastAPI
import uvicorn

# Import the app from above
# For Colab, we'll create a minimal app here
app = FastAPI()

@app.get('/api/health')
async def health_check():
    return {'status': 'ok'}

@app.get('/')
async def root():
    return {'message': 'NGS Pipeline Server Running'}

if __name__ == '__main__':
    uvicorn.run(app, host='127.0.0.1', port=7860, log_level='info')
'''

# Write the server script
with open('/content/ngs_pipeline/server.py', 'w') as f:
    f.write(server_script)

# Start the server in background
def run_server():
    subprocess.run(['bash', '-c', 
                   'source /usr/local/miniconda/etc/profile.d/conda.sh && '
                   'conda activate ngs_env && '
                   'cd /content/ngs_pipeline && '
                   'python server.py'],
                  stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

# Wait for server to start
time.sleep(3)

print('✓ FastAPI server started on http://127.0.0.1:7860')

## 9. Get Public URL via Ngrok

In [ ]:
import requests
import time

# Function to get ngrok URL
def get_ngrok_url():
    for i in range(10):
        try:
            response = requests.get('http://127.0.0.1:4040/api/tunnels')
            tunnels = response.json()['tunnels']
            if tunnels:
                return tunnels[0]['public_url']
        except:
            time.sleep(1)
    return None

# Get URL
public_url = get_ngrok_url()

if public_url:
    print('\n' + '='*70)
    print('✓ NGS PIPELINE IS NOW RUNNING!')
    print('='*70)
    print(f'Public URL: {public_url}')
    print(f'Local URL:  http://127.0.0.1:7860')
    print('='*70)
    print(f'\nAccess the web interface at: {public_url}')
    print('Keep this notebook running to maintain the connection.\n')
else:
    print('⚠ Still waiting for ngrok tunnel... Please wait a moment.')

## 10. Run a Sample Pipeline Stage (Optional)

In [ ]:
# Example: Run a pipeline stage
# This is just an example - replace with actual SRA accession and configuration

import subprocess
import os

def run_pipeline_stage(project_name, stage_name, sra_accession=None):
    """
    Run a specific pipeline stage for a project
    
    Args:
        project_name: Name of the project
        stage_name: Name of the pipeline stage
        sra_accession: SRA accession ID (for getting_data stage)
    """
    pipeline_dir = '/content/ngs_pipeline'
    
    cmd = f'bash -c "source /usr/local/miniconda/etc/profile.d/conda.sh && '
    cmd += f'conda activate ngs_env && '
    cmd += f'cd {pipeline_dir} && '
    cmd += f'python pipeline_runner.py --project {project_name} --stage {stage_name}'
    
    if sra_accession:
        cmd += f' --sra {sra_accession}'
    
    cmd += '"'
    
    print(f'Running stage: {stage_name} for project: {project_name}')
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    
    if result.returncode == 0:
        print(f'✓ Stage completed successfully')
        if result.stdout:
            print('Output:', result.stdout[:500])
    else:
        print(f'✗ Stage failed')
        if result.stderr:
            print('Error:', result.stderr[:500])

# Example usage (uncomment to run):
# run_pipeline_stage('my_project', 'getting_data', 'SRR12345678')

print('Pipeline runner function ready. Use run_pipeline_stage() to execute stages.')

## 11. Keep Server Running

In [ ]:
import time
import requests

# Monitor the server
print('\n' + '='*70)
print('NGS PIPELINE SERVER STATUS')
print('='*70)

try:
    # Check local server
    response = requests.get('http://127.0.0.1:7860/api/health', timeout=5)
    if response.status_code == 200:
        print('✓ FastAPI Server: RUNNING')
    else:
        print('✗ FastAPI Server: ERROR')
except:
    print('✗ FastAPI Server: NOT RESPONDING')

try:
    # Check ngrok tunnel
    response = requests.get('http://127.0.0.1:4040/api/tunnels', timeout=5)
    tunnels = response.json()['tunnels']
    if tunnels:
        print('✓ Ngrok Tunnel: ACTIVE')
        print(f'  Public URL: {tunnels[0]["public_url"]}')
    else:
        print('✗ Ngrok Tunnel: NO TUNNELS FOUND')
except:
    print('✗ Ngrok Tunnel: NOT RESPONDING')

print('='*70)
print('\nRun this cell periodically to check the server status.')
print('The server will remain active as long as this notebook is running.\n')